# Step 5 — Does a smarter model beat the fixed EAR threshold?

`step_02` built a working blink detector from one fixed rule: threshold =
0.75 × (90th percentile EAR). It looked convincing on its own plot. Now we
check it properly, against **real hand-labelled ground truth** (from
`step_04`), and compare it to a small logistic regression model that also
sees gaze and head context.

**Make a prediction before running anything:** do you expect the model to
beat the simple rule here? Why or why not? There's no wrong answer to
write down — the point is to have a prediction to check against what
actually happens.


## At a glance

| | |
|---|---|
| **Workshop step** | 5 of 6 |
| **Needs in this folder** | for each blink clip: `features_blinks_<name>.csv` (from step 3) and its label file `labels_blinks_<name>__<labeller>.csv` (from step 4, blink mode). In a group, one pair per person. |
| **Produces** | nothing saved to disk. Prints precision, recall and F1 for both methods. |
| **Run time** | a few seconds |
| **Next** | `step_06_gaze_classifier.ipynb` |

**How to run:** in the menu choose *Run > Restart Kernel and Run All Cells*, or press Shift+Enter on each cell from the top down. Do not skip cells: later cells use variables made by earlier ones.

If the first code cell stops with "No labelled blink clips found", you have not done step 4 in blink mode yet. If it stops because one clip has several label files, set `LABELLER` in that cell to choose one.

**Working in a group:** this notebook uses every labelled blink clip it finds, so the model can be trained on more than one person. See `TEST_CLIP` in the first code cell.

## Precision, recall, F1 — in plain terms

For each method, "closed" is the thing we're trying to catch.

- **Precision**: of the frames the method *called* closed, what fraction
  really were? (High precision = few false alarms.)
- **Recall**: of the frames that *really were* closed, what fraction did
  the method catch? (High recall = few misses.)
- **F1**: a single balanced score — you can't raise F1 by sacrificing one
  of these for the other, it rewards doing reasonably well on both.

Which matters more depends on the application. For a blink-controlled
device, is a missed blink (low recall) or a false blink (low precision)
the worse failure? Worth deciding *before* you see numbers, not after.


## Load the hand labels and the features

Each clip has two files: the features (numbers per frame, from step 3) and
the labels (what a person saw, from step 4). We load every clip that has
both.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# TRY IT (groups): which clip is held back for testing?
#   None             every clip is cut in time: the model trains on the
#                    first part of each clip and is tested on the rest.
#   "blinks_<name>"  that whole clip is held back. The model trains on
#                    everyone else's clips and is tested on a person it
#                    has never seen. Needs at least two labelled clips.
# Run the notebook once per person in your group as the held-out clip.
# Is one person harder to predict than the others? Why might that be?
# ---------------------------------------------------------------------
TEST_CLIP = None

# Only needed if one clip was labelled by several people: put the labeller's
# name here to choose whose labels to use. None = use the only file present.
LABELLER = None


def find_label_file(clip_name):
    """Return the label file for one clip, or None if it is not labelled yet."""
    matches = sorted(glob.glob(f"labels_{clip_name}__*.csv"))
    if len(matches) > 1 and LABELLER:
        matches = [m for m in matches if m == f"labels_{clip_name}__{LABELLER}.csv"]
    if len(matches) > 1:
        raise SystemExit(f"{clip_name} has {len(matches)} label files: {matches}. Set LABELLER to pick one.")
    return matches[0] if matches else None


def stamp_frame_labels(intervals, n_frames):
    """Turn labelled intervals (start frame, end frame) into one label per frame."""
    labels = np.full(n_frames, "open", dtype=object)
    for _, row in intervals.iterrows():
        labels[int(row.start_frame):int(row.end_frame) + 1] = row.label
    return labels


clips = {}   # clip name -> table with one row per frame (features + label)
for feat_path in sorted(glob.glob("features_blinks_*.csv")):
    clip_name = os.path.basename(feat_path)[len("features_"):-len(".csv")]
    labels_path = find_label_file(clip_name)
    if labels_path is None:
        print(f"Skipping {clip_name}: no label file yet (step 4, blink mode)")
        continue
    feats = pd.read_csv(feat_path)
    intervals = pd.read_csv(labels_path)
    feats["label"] = stamp_frame_labels(intervals, len(feats))
    feats["clip"] = clip_name
    clips[clip_name] = feats
    n_closed = (feats["label"] == "closed").sum()
    print(f"{clip_name}: {len(feats)} frames, {n_closed} labelled closed (labels from {labels_path})")

if not clips:
    raise SystemExit("No labelled blink clips found. Run step 3, then step_04_label_intervals.py with MODE = 'blink'.")

### One label per frame

A label file stores intervals (start frame, end frame). The model needs one label for every frame, so `stamp_frame_labels` above stamps each interval onto its frames. Here is the result for the first clip:

In [ ]:
first_clip = next(iter(clips))
print(first_clip)
clips[first_clip]["label"].value_counts()

## Giving the model a notion of "change", not just a snapshot

A single EAR value can't tell "closing" apart from "steady, just looking
down" — both can be a low number. Two extra features help:
- `ear_delta`: how much EAR changed from the previous frame (closing shows
  up as a fast *drop*, not just a low value)
- `ear_min3`: the smallest EAR in a short 3-frame window around this one

This is context the fixed threshold rule never sees at all.


In [ ]:
# ---------------------------------------------------------------------
# TRY IT: FEATURE_COLS is everything the model is allowed to look at.
#   - Remove the gaze and head columns (keep only the three "ear" ones).
#     Does the score drop? If not, were they helping at all?
#   - Keep ONLY "ear". The model then knows what the fixed threshold
#     knows. Does it still beat the threshold?
#   - EAR_MIN_WINDOW is how many neighbouring frames "ear_min" looks at.
#     Try 5 or 7.
#   - Harder: add your own feature below, for example each clip's EAR
#     divided by that clip's median EAR, so that people with different
#     eye shapes look more alike to the model.
# ---------------------------------------------------------------------
EAR_MIN_WINDOW = 3
FEATURE_COLS = ["ear", "ear_delta", "ear_min3", "gaze_x", "gaze_y", "nose_dx", "nose_dy"]

for clip_name, feats in clips.items():
    # computed one clip at a time, so the last frame of one clip is never
    # compared with the first frame of the next
    feats["ear_delta"] = feats["ear"].diff().fillna(0.0)
    feats["ear_min3"] = feats["ear"].rolling(EAR_MIN_WINDOW, center=True, min_periods=1).min()

    # frames where no face was found have empty (NaN) features. The model
    # cannot use them, so they are left out and counted.
    n_before = len(feats)
    clips[clip_name] = feats.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    print(f"{clip_name}: dropped {n_before - len(clips[clip_name])} frames without a face")

## Splitting train/test — why not shuffle?

Neighbouring frames of the same blink are near-identical. A random 80/20
shuffle would let some frames of one blink land in training and others (a
few milliseconds later, basically the same) land in test. That inflates
the score without the model learning anything general. There are two
honest ways to split, and `TEST_CLIP` in the first code cell chooses:

- **`TEST_CLIP = None`: split each clip in time.** Train on the first part
  of every clip, test on the later part, which the model has never seen
  *any* frame from. This works with a single clip. It is the weaker test:
  the model has already seen the same person, camera and lighting.
- **`TEST_CLIP = "blinks_<name>"`: hold out a whole clip.** Train on the
  other clips, test on this one. With one clip per person this asks the
  question that matters for a real device: *does it work on someone it was
  not trained on?* Expect lower scores than the split in time.

In [ ]:
# TRY IT: the share of each clip used for training when TEST_CLIP is None.
# Try 0.5 and 0.9. With 0.9 the test part is tiny: how much do you trust a
# score that comes from two or three blinks?
TRAIN_FRACTION = 0.7

if TEST_CLIP is None:
    train_parts, test_parts = [], []
    for clip_name, feats in clips.items():
        split = int(len(feats) * TRAIN_FRACTION)
        train_parts.append(feats.iloc[:split])
        test_parts.append(feats.iloc[split:])
    print(f"Split in time: first {TRAIN_FRACTION:.0%} of each clip for training, the rest for testing")
else:
    if TEST_CLIP not in clips:
        raise SystemExit(f"TEST_CLIP = {TEST_CLIP!r} is not a labelled clip. Choose from {list(clips)}.")
    if len(clips) < 2:
        raise SystemExit("Holding out a whole clip needs at least two labelled clips. Set TEST_CLIP = None.")
    train_parts = [feats for clip_name, feats in clips.items() if clip_name != TEST_CLIP]
    test_parts = [clips[TEST_CLIP]]
    print(f"Held-out clip: testing on {TEST_CLIP}, training on the other {len(train_parts)} clip(s)")

train = pd.concat(train_parts, ignore_index=True)
test = pd.concat(test_parts, ignore_index=True)
for part_name, part in [("Train", train), ("Test ", test)]:
    per_clip = ", ".join(f"{c}: {(g.label == 'closed').sum()} closed of {len(g)}" for c, g in part.groupby("clip"))
    print(f"{part_name}: {len(part)} frames ({per_clip})")

# the model needs both classes to learn from, and the test part needs some blinks to score
if train["label"].nunique() < 2 or (test["label"] == "closed").sum() == 0:
    raise SystemExit("The train part needs open AND closed frames, and the test part needs some closed frames. "
                     "The blinks are probably bunched together: label more blinks, or record a clip with blinks spread across it.")

## Method 1: the fixed threshold from `step_02`

The rule is the same as in `step_02`: closed means EAR below 0.75 x the 90th percentile. Each clip in the test part gets its own threshold, because the open-eye EAR differs between people and recordings.

The threshold printed below will not match the one you saw in `step_02`. When a clip is split in time, it is computed over the test part only (the last 30%), not over the whole clip.

In [ ]:
def report(name, y_true, y_pred):
    p = precision_score(y_true, y_pred, pos_label="closed", zero_division=0)
    r = recall_score(y_true, y_pred, pos_label="closed", zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label="closed", zero_division=0)
    print(f"  {name:<22} precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}  "
          f"(closed frames: true={sum(y_true == 'closed')}, predicted={sum(y_pred == 'closed')})")
    return f1


# TRY IT: use the values you liked best in step_02. Does the setting that
# looked best on the plot there also give the best F1 against your labels?
BASELINE_PERCENTILE = 90
THRESHOLD_FACTOR = 0.75

test["base_pred"] = "open"
print("Baseline (step_02 rule), one threshold per clip:")
for clip_name, part in test.groupby("clip"):
    threshold = THRESHOLD_FACTOR * np.nanpercentile(part["ear"], BASELINE_PERCENTILE)
    test.loc[part.index, "base_pred"] = np.where(part["ear"] < threshold, "closed", "open")
    print(f"  {clip_name}: threshold={threshold:.3f}")
base_f1 = report("fixed threshold", test["label"].values, test["base_pred"].values)

## Method 2: logistic regression

In [ ]:
# The scaler puts every feature on the same scale (mean 0, spread 1). It is
# fitted on the training part only: the test part must stay unseen.
scaler = StandardScaler().fit(train[FEATURE_COLS])
X_train = scaler.transform(train[FEATURE_COLS])
X_test = scaler.transform(test[FEATURE_COLS])

# TRY IT: class_weight="balanced" tells the model that the rare "closed"
# frames matter as much as the many "open" ones. Replace it with None and
# watch precision and recall move in opposite directions. Which setting
# would you want for a blink-controlled device?
clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(X_train, train["label"])
test["model_pred"] = clf.predict(X_test)
print("Model: logistic regression on EAR + temporal context + gaze/head features")
model_f1 = report("logistic regression", test["label"].values, test["model_pred"].values)

print(f"\n{'Model beats the fixed threshold' if model_f1 > base_f1 else 'Model does NOT beat the fixed threshold'} "
      f"(f1 {model_f1:.3f} vs {base_f1:.3f}).")

# with more than one clip in the test part, show each one separately:
# a pooled score can hide one clip doing much worse than the others
if test["clip"].nunique() > 1:
    for clip_name, part in test.groupby("clip"):
        print(f"\n{clip_name}:")
        report("fixed threshold", part["label"].values, part["base_pred"].values)
        report("logistic regression", part["label"].values, part["model_pred"].values)

## Discuss

- Did your prediction from the top of this notebook match what happened?
- Whichever method "won", **how much do you trust that result?** How
  many true closed frames were in your test set (see the printed count
  above)? With so few, would you expect the same winner on a different
  clip, or a different split?
- In a group: compare the split in time (`TEST_CLIP = None`) with holding
  out each person in turn. Which gives the lower score, and which would
  you report if someone asked "does this work on a new user"?
- If the model lost: is that because it's genuinely worse, or could the
  gap be about *how blinks were labelled* in the first place? Did everyone
  in your group use the same definition of "closed"? What evidence would
  tell these two explanations apart? (`STUDENT_TASKS.md` task 1b.)

Look back through the code cells for the `TRY IT` comments: each one marks
a value that is yours to change.

Related open tasks if this interests you: `STUDENT_TASKS.md` tasks 1a
(more clips), 1b (agreement between labellers), and 2 (a non-ML dip
detector as a third comparison point).